# 04. Alignments (окно с 2013-01-01)

Отдельный ноутбук, чтобы самый долгий расчёт шёл независимо от остальных. Заменяет `run_split.py`, вторую половину `petri_states_eval_*.ipynb`, `retry_timeouts.py`, `fix_timeouts_051126.py` и `im_timeouts.py`.

**Сначала выполнить `03_tbr_precision.ipynb`.** Этот ноутбук в обеих частях можно запускать одновременно: процессы лёгкие, журнал они не строят, папки результатов разные.

Как идёт расчёт (`common/align_parallel.py`): свободный процесс берёт следующий вариант трассы любой модели; каждый результат сразу записан. *Kernel → Interrupt* останавливает всё, запуск ячейки заново продолжает с места.

Лимит на вариант — **120 с** (как в прежнем расчёте этой части). Лимит считается по настенным часам: если процессор перегружен, тайм-аутов будет больше, поэтому `WORKERS` не стоит ставить выше числа свободных ядер.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "common"))
import settings as cfg

PART = "2013"                         # "2013" — с окном, "all" — без окна

import pandas as pd
import align_parallel as ap

WORKERS = cfg.N_WORKERS                  # процессов в этом ноутбуке; см. комментарий в settings.py
CONFIGS = cfg.EVAL_CONFIGS               # если их несколько, считаются вперемешку одним набором процессов
SCRIPT = cfg.COMMON / "align_parallel.py"
FIX = cfg.COMMON / "fix_timeouts.py"

## Шаг 1. Alignments на всём test, лимит 120 с

In [2]:
ap.launch([SCRIPT, PART, "--configs", *CONFIGS, "--workers", WORKERS])

D:\Git\diploma\.venv\Scripts\python.exe -u D:\diploma\jupyter lab\петри_пересчет\common\align_parallel.py 2013 --configs long180_v2 long180_v1 long365_v1 --workers 3


23:38:14 часть 2013: заданий 9, процессов 3


23:38:14 проверка easy soundness: 9 сетей (лимит 600 с на сеть)


23:38:17   long180_v1/DFG_p0.05: easy_sound = True


23:38:17   long180_v1/HM_dep0.95: easy_sound = True


23:38:17   long180_v1/DFG_p0.0: easy_sound = True


... (125 outputs left out)

00:02:07 long365_v1/HM_dep0.95 готово: fitness 0.9442 | тайм-аут трасс 0 из 2255


00:02:07 long365_v1/SplitMiner готово: fitness 0.6334 | тайм-аут трасс 0 из 2255


00:02:07 long365_v1/IM_noise0.2 готово: fitness 0.9998 | тайм-аут трасс 0 из 2255


00:02:07 ALIGN_DONE, 23.9 мин


## Шаг 2. Досчёт тайм-аутов, LONG = 180

DFG p0.05 — точный расчёт (минуты); Split Miner — лимит 1800 с; IM 0.2 — варианты, идеальные по TBR, и лимит 600 с на остальные.

Итог: `out/long180_v2/split_eval_timeouts_fixed.csv`.

In [3]:
for config in CONFIGS:
    if config.startswith("long180"):
        ap.launch([FIX, PART, f"--workers={WORKERS}", f"--config={config}"])

D:\Git\diploma\.venv\Scripts\python.exe -u D:\diploma\jupyter lab\петри_пересчет\common\fix_timeouts.py 2013 --workers=3 --config=long180_v2


00:02:11 2013: вариантов в test 1,150


00:02:11 2013 DFG_p0.05: тайм-аутов 0 вариантов (0 трасс)


00:02:11 шаг dfg: 0.0 мин


experiment       model  timeout_before  timeout_after  fitness_align_before  fitness_align_after  f1_align_before  f1_align_after


      2013   DFG_p0.05               0              0                0.9652               0.9652           0.8482          0.8482


... (50 outputs left out)

      2013  SplitMiner               0              0                0.6025               0.6025           0.6787          0.6787


      2013 IM_noise0.2               0              0                0.9998               0.9998           0.5378          0.5378


00:02:14 итог -> split_eval_timeouts_fixed.csv


FIX_DONE


## Шаг 3. Досчёт тайм-аутов варианта 1, `long365_v1`, лимит 600 с

Выполняется только при `RUN_V1 = True`.

Варианты, не уложившиеся в лимит шага 1, для моделей из `cfg.PARTS[PART]["retry_models"]`. Итог: `out/long365_v1/split_eval_retry*s.jsonl`.

In [4]:
if cfg.RUN_V1:
    ap.launch([SCRIPT, PART, "--configs", "long365_v1", "--retry", "--workers", WORKERS])
else:
    print("шаг выключен: RUN_V1 = False в common/settings.py")

D:\Git\diploma\.venv\Scripts\python.exe -u D:\diploma\jupyter lab\петри_пересчет\common\align_parallel.py 2013 --configs long365_v1 --retry --workers 3


00:02:14 часть 2013: заданий 3, процессов 3


00:02:14   long365_v1/DFG_p0.05: вариантов 1,143, уже готово 0, осталось 0, лимит 600 с


00:02:14   long365_v1/SplitMiner: вариантов 1,143, уже готово 0, осталось 0, лимит 600 с


00:02:14   long365_v1/IM_noise0.2: вариантов 1,143, уже готово 0, осталось 0, лимит 600 с


00:02:14 long365_v1/DFG_p0.05 готово: fitness 0.9635 | тайм-аут трасс 0 из 2255


00:02:14 long365_v1/SplitMiner готово: fitness 0.6334 | тайм-аут трасс 0 из 2255


00:02:14 long365_v1/IM_noise0.2 готово: fitness 0.9998 | тайм-аут трасс 0 из 2255


00:02:14 ALIGN_DONE, 0.0 мин


## Шаг 4. Сколько посчитано

Можно запускать в любой момент из отдельной консоли ноутбука: читает только контрольные точки.

In [5]:
rows = []
for config in CONFIGS:
    keys, counts = ap.load_variants(PART, config)
    d = cfg.out_dir(PART, config)
    finished = {r["model"]: r for r in ap.jl_read(d / "split_eval.jsonl")}
    for name in cfg.EVAL_MODELS[config]:
        ck = ap.jl_read(d / "align_ckpt" / f"{name}_align.jsonl")
        rows.append({"конфигурация": config, "модель": name, "вариантов": len(keys), "готово": len(ck),
                     "тайм-аутов (варианты)": sum(r["timeout"] for r in ck), "модель закончена": name in finished,
                     "fitness": finished.get(name, {}).get("fitness_align_test"), "часов процессора": round(sum(r["sec"] for r in ck) / 3600, 2)})
pd.DataFrame(rows).round(4)

,конфигурация,модель,вариантов,готово,тайм-аутов (варианты),модель закончена,fitness,часов процессора
0,long180_v2,DFG_p0.05,1150,1150,0,True,0.9652,0.01
1,long180_v2,DFG_p0.0,1150,1150,0,True,1.0000,0.00
2,long180_v2,HM_dep0.95,1150,1150,0,True,0.9437,0.01
3,long180_v2,SplitMiner,1150,1150,0,True,0.6386,0.01
4,long180_v2,IM_noise0.2,1150,1150,0,True,0.9998,0.57
5,long180_v1,DFG_p0.05,1150,1150,0,True,0.9652,0.01
6,long180_v1,DFG_p0.0,1150,1150,0,True,1.0000,0.00
7,long180_v1,HM_dep0.95,1150,1150,0,True,0.9437,0.01
8,long180_v1,SplitMiner,1150,1150,0,True,0.6025,0.01
9,long180_v1,IM_noise0.2,1150,1150,0,True,0.9998,0.57
